# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [21]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

In [22]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [37]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# class Website:
#     """
#     A utility class to represent a Website that we have scraped, now with links
#     """

#     def __init__(self, url):
#         self.url = url
#         response = requests.get(url, headers=headers)
#         self.body = response.content
#         soup = BeautifulSoup(self.body, 'html.parser')
#         self.title = soup.title.string if soup.title else "No title found"
#         if soup.body:
#             for irrelevant in soup.body(["script", "style", "img", "input"]):
#                 irrelevant.decompose()
#             self.text = soup.body.get_text(separator="\n", strip=True)
#         else:
#             self.text = ""
#         links = [link.get('href') for link in soup.find_all('a')]
#         self.links = [link for link in links if link]

#     def get_contents(self):
#         return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

class Website:
    """
    A utility class to represent a Website that we have scraped.
    It supports both JavaScript-rendered SPAs (React, Angular, etc.) and static HTML pages.
    """

    def __init__(self, url):
        self.url = url

        # Try loading with Selenium first
        try:
            options = Options()
            options.add_argument('--headless')
            options.add_argument('--disable-gpu')
            options.add_argument('--no-sandbox')
            driver = webdriver.Chrome(options=options)
            driver.get(url)
            time.sleep(5)  # Wait for JS to render
            self.body = driver.page_source
            driver.quit()
        except WebDriverException:
            # Fall back to requests if Selenium fails (likely a static site)
            response = requests.get(url, headers=headers)
            self.body = response.content

        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"

        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""

        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link and not link.startswith('#') and not link.startswith('mailto') 
                      and not link.startswith('tel')and not link.startswith('/mailto') 
                      and not link.startswith('editfornt')]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [38]:
ed = Website("https://edwarddonner.com")
# to get or extract links presents within the wexbsite
# ed.links
print(ed.get_contents())

Webpage Title:
Home - Edward Donner
Webpage Contents:
Skip to content
Home
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy DJing (but I’m badly out of practice), amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. Recruiters use our product today to source, understand, engage and manage talent. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
We work with groundbreaking, proprietary LLMs verticalized for talent, we’ve
patented
our matching model, and our award-winning platform has h

In [39]:
edKo = Website("https://kovad.net/")
edKo.links

['https://wa.me/2347034849938',
 '/index-2.html',
 '/',
 '/contact',
 '/about_us',
 'https://wa.me/2347034849938',
 'https://wa.me/2347034849938',
 'services.html',
 'services.html',
 'services.html',
 'services.html',
 'services.html',
 'services.html',
 'services.html',
 'services.html',
 'index-2.html',
 'https://wa.me/2347034849938',
 'https://wa.me/2347034849938',
 'https://wa.me/2347034849938',
 '/',
 '/about_us',
 '/contact',
 '/',
 '/about_us']

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [48]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"},
        {"type": "documentation page", "url": "https://another.full.url/docs"}
        {"type": "chatroom page", "url": "https://another.full.url/chat"}
        {"type": "data Sets page", "url": "https://another.full.url/datasets"}
    ]
}
"""

In [49]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"},
        {"type": "documentation page", "url": "https://another.full.url/docs"}
        {"type": "chatroom page", "url": "https://another.full.url/chat"}
        {"type": "data Sets page", "url": "https://another.full.url/datasets"}
    ]
}



In [50]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [51]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/
https://edwarddo

In [52]:
print(get_links_user_prompt(edKo))

Here is the list of links on the website of https://kovad.net/ - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://wa.me/2347034849938
/index-2.html
/
/contact
/about_us
https://wa.me/2347034849938
https://wa.me/2347034849938
services.html
services.html
services.html
services.html
services.html
services.html
services.html
services.html
index-2.html
https://wa.me/2347034849938
https://wa.me/2347034849938
https://wa.me/2347034849938
/
/about_us
/contact
/
/about_us


In [53]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        # this code below will only work if you mention it in your project
        # that response should be in json then the code below is available for openai not for claude.ai
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [56]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/openai/gpt-oss-120b',
 '/openai/gpt-oss-20b',
 '/Qwen/Qwen-Image',
 '/tencent/Hunyuan-1.8B-Instruct',
 '/rednote-hilab/dots.ocr',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/Qwen/Qwen-Image',
 '/spaces/black-forest-labs/FLUX.1-Krea-dev',
 '/spaces/Qwen/Qwen3-Coder-WebDev',
 '/spaces/Wan-AI/Wan-2.2-5B',
 '/spaces',
 '/datasets/fka/awesome-chatgpt-prompts',
 '/datasets/nvidia/Nemotron-Post-Training-Dataset-v1',
 '/datasets/spatialverse/InteriorGS',
 '/datasets/HuggingFaceH4/Multilingual-Thinking',
 '/datasets/spatialverse/InteriorAgent',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google',
 '/Intel',
 '/microsoft',
 '/grammarly',
 '/Writer',
 '/docs/transformers',


In [57]:
get_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'models page', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [68]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        if link["url"] and not link["url"].startswith('#') and not link["url"].startswith('mailto') \
        and not link["url"].startswith('tel')and not link["url"].startswith('/mailto') \
        and not link["url"].startswith('editfornt'):
            result += f"\n\n{link['type']}\n"
            result += Website(link["url"]).get_contents()
    return result

In [69]:
print(get_all_details("https://huggingface.co"))

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'documentation page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}]}
Landing page:
Webpage Title:
Hugging Face – The AI community building the future.
Webpage Contents:
Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 1M+ model

In [71]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [72]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [73]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'homepage', 'url': 'https://huggingface.co/'}, {'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'documentation page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'community page', 'url': 'https://discuss.huggingface.co'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\nLanding page:\nWebpage Title:\nHugging Face – The AI community building the future.\nWebpage Contents:\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 1M+ models\nTrending on\nthis week\nModels\nopenai/gpt-oss-120b\nUpdated\nabout 3 hours ago\n•\n238k\n•\n2.99k\nopenai/gpt-oss-20b\nUpdated\nabout 3 hours ago\n•\n864k\n•\n2.55k\nQwen/Qwen-Image\nUpdated\n3 days ago\n•\n42.3k\n•\n1.32k\ntencent/Hunyuan-1.8B-Instruct\nUpdated\n3 days ago\n•\n2.29k\n•\n554\nrednote-hilab/dots.ocr\nUpdated\n1 day ago\n•\n10.6k\n•\n511\nBrowse 1M+ models\nSpaces\nRunning\non\nZero\n350\n350\nQwen Image\

In [74]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [75]:
create_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'docs page', 'url': 'https://huggingface.co/docs'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'GitHub page', 'url': 'https://github.com/huggingface'}, {'type': 'status page', 'url': 'https://status.huggingface.co/'}, {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}, {'type': 'LinkedIn page', 'url': 'https://www.linkedin.com/company/huggingface/'}]}


# Hugging Face Brochure

## Who We Are
**Hugging Face** is a leading AI community dedicated to building the future of machine learning. Our platform serves as the collaborative hub where developers, researchers, and enthusiasts converge to create, discover, and share models, datasets, and applications.

---

## Our Vision
At Hugging Face, we believe in the power of collaboration to accelerate innovation. As the home of machine learning, we empower users to: 
- **Create** and share unlimited public models and datasets.
- **Discover** the latest AI applications from a community-driven directory.
- **Explore** diverse modalities including text, image, video, audio, and 3D.

---

## Community Impact
With over **50,000 organizations** using our services, Hugging Face has become a trusted partner for companies such as:
- **Google**
- **Microsoft**
- **Amazon**
- **Intel**
- **Grammarly**

Our community is continuously evolving, featuring thousands of models (1M+) and datasets (250k+) that are easily accessible and collaborative.

---

## Products & Services
We offer a robust suite of tools tailored for every ML professional:
- **Models**: Access the latest and trending machine learning models. 
- **Datasets**: Share and utilize a wide variety of datasets to advance your projects.
- **Spaces**: Host and run applications easily in an optimized environment.
- **Enterprise Solutions**: Tailored for teams needing advanced features including security, dedicated support, and more, starting at **$20/user/month**.

---

## Company Culture
At Hugging Face, inclusivity and collaboration are at the heart of our culture. We encourage innovation, a passion for learning, and a commitment to solving challenging problems in machine learning. Our team thrives in an environment that supports growth, creativity, and teamwork.

---

## Join Us
Explore career opportunities at Hugging Face! We are always seeking talented individuals who are passionate about AI and eager to make a meaningful impact in the ML community. Check our [Jobs Page](#) for current openings and be a part of our journey to build the future of AI.

---

## Get In Touch
For more information about our offerings, partnerships, and community initiatives, visit our website or follow us on social media:

- [Website](https://huggingface.co)
- [Twitter](https://twitter.com/huggingface)
- [LinkedIn](https://linkedin.com/company/huggingface)
- [GitHub](https://github.com/huggingface)

---

**Join us at Hugging Face, where we shape the future of machine learning together!**

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [76]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        # below code is use to make typewriter feature enable when returning the response
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [77]:
stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'documentation page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'models page', 'url': 'https://huggingface.co/models'}, {'type': 'datasets page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}]}


# Hugging Face Brochure

**The AI Community Building the Future**

---

## About Hugging Face

Hugging Face is a pioneering platform dedicated to fostering collaboration within the machine learning community. We offer a space where developers, data scientists, and enthusiasts can come together to explore, create, and share over **1 million models** and **250,000 datasets**. Our mission is to make machine learning more accessible and collaborative, leading the charge in AI innovation.

---

## Our Platform

### Key Features:

- **Models & Datasets**: Access and contribute to an extensive library of pre-trained models and datasets.
- **Spaces**: Build and share applications without the heavy lifting, using our user-friendly interface to deploy models with ease.
- **Community Collaboration**: Engage with a vibrant community that is continuously pushing the bounds of what's possible in AI. Collaborate on projects, follow the latest research, and learn from shared experiences.

### Enterprise Solutions:

Hugging Face also provides enterprise-grade solutions tailored for businesses looking to leverage AI effectively and securely. Our offerings include:

- **Compute Solutions**: Optimized inference endpoints to enhance the deployment of your models.
- **Security & Support**: Robust access controls, dedicated support, and enterprise functionalities designed for large teams and organizations.

---

## Company Culture

At Hugging Face, we pride ourselves on fostering an inclusive and supportive environment where every voice is heard. We are committed to:

- **Open Source Philosophy**: Contributing to a collaborative ecosystem in AI development.
- **Innovation**: Encouraging creativity and exploration across all levels of the organization.
- **Community Engagement**: Empowering our community through forums, blogs, and educational resources.

Join a team that values collaboration, knowledge sharing, and the power of AI technology to drive positive change.

---

## Who We Work With

More than **50,000 organizations** globally trust Hugging Face, including industry leaders such as:

- **Meta**
- **Google**
- **Microsoft**
- **Amazon**

Our diverse clientele reflects the credibility and reliability of our solutions in various sectors including tech, education, and research.

---

## Career Opportunities

Are you passionate about AI and machine learning? Join our team at Hugging Face! We offer a range of job positions that encourage innovation and personal growth. Our team is filled with knowledgeable individuals eager to make a difference in the AI landscape.

- **Current Openings**: Explore our job listings to find positions that align with your skills and career aspirations.
- **Culture and Benefits**: We value our employees and offer competitive salaries, flexible working conditions, and a diverse culture that nurtures growth.

---

## Connect With Us

Stay in touch and join our community:

- **Website**: [Hugging Face](https://huggingface.co)
- **Social Media**: 
  - GitHub
  - Twitter
  - LinkedIn
  - Discord

Become a part of the journey towards building the future of AI with Hugging Face!

--- 

For further details on our offerings and community contributions, visit our website or sign up today to start exploring AI possibilities!

In [78]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Found links: {'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'}, {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'}, {'type': 'documentation page', 'url': 'https://huggingface.co/docs'}, {'type': 'blog page', 'url': 'https://huggingface.co/blog'}, {'type': 'model page', 'url': 'https://huggingface.co/models'}, {'type': 'dataset page', 'url': 'https://huggingface.co/datasets'}, {'type': 'spaces page', 'url': 'https://huggingface.co/spaces'}, {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'}, {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'}]}


# Hugging Face Brochure

---

## **About Us**

### **Hugging Face**
Hugging Face is at the forefront of the artificial intelligence revolution, acting as a community-driven platform where the machine learning community collaborates on models, datasets, and applications. Our mission is to democratize AI and responsibly build the future through innovative tools and resources.

---

## **What We Offer**

### **AI Models**
Explore our extensive library of **1M+ AI models**, from text generation to image synthesis, allowing easy access to state-of-the-art technology. Some of our trending models include:
- **openai/gpt-oss-120b**
- **openai/gpt-oss-20b**
- **Qwen/Qwen-Image**

### **Datasets**
Access over **250,000 datasets** specifically curated for a variety of machine learning tasks, ensuring that you have the resources needed for your projects.

### **Spaces**
Engage with our community by running and sharing interactive applications on our platform. Explore a collection of **400k+ applications** built by users around the world.

### **Enterprise Solutions**
Our enterprise offerings include advanced security measures, dedicated support, and optimized computing resources to accelerate your AI initiatives.

---

## **Customers**

Hugging Face proudly supports over **50,000 organizations**, including industry leaders such as:
- Amazon
- Google
- Microsoft
- Meta

Our platform is a trusted resource for teams at both enterprise and non-profit organizations, facilitating collaboration and innovation across the AI landscape.

---

## **Company Culture**

At Hugging Face, we foster an inclusive and collaborative environment where creativity thrives. We believe in open-source principles, engaging with our community to build foundational tools for machine learning. Our commitment to responsible AI development is reflected in our code of conduct and community guidelines.

---

## **Careers at Hugging Face**

We are always looking for motivated individuals to join our dynamic team. Whether you're an engineer, researcher, marketer, or designer, we want to hear from you! We offer:
- Competitive salaries and benefits.
- Opportunities for career growth in a rapidly evolving field.
- A supportive workplace culture that values diversity and innovation.

Explore current job openings [here](https://huggingface.co/join).

---

## **Join the Future of AI!**

Be part of a vibrant community that is shaping the landscape of artificial intelligence. Whether you're looking to leverage cutting-edge technology, collaborate with like-minded individuals, or explore career opportunities, **Hugging Face** is the platform for you.

---

### **Connect with Us**
- [Website](https://huggingface.co)
- [GitHub](https://github.com/huggingface)
- [Twitter](https://twitter.com/huggingface)
- [LinkedIn](https://linkedin.com/company/huggingface)

---

**Hugging Face - The AI community building the future.**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>